# 10 — Inventory Optimization

## Sales Order, Backlog & Inventory Analytics

### Objective

This notebook converts inventory, demand, and ABC–XYZ insights into actionable inventory-optimization decisions.

### Analysis Scope

The analysis will focus on:

- Current inventory position
- Demand-based inventory coverage
- Safety-stock adequacy
- Reorder-point adequacy
- Stockout risk
- Excess and overstock exposure
- Inventory gap and surplus quantities
- ABC–XYZ-based inventory prioritization
- High-value inventory risk
- Replenishment priority
- Inventory optimization opportunities
- Working-capital reduction opportunities
- Product-level recommended inventory actions

### Goal

The objective is to identify where inventory should be:

- Replenished
- Protected
- Maintained
- Reduced
- Closely monitored

while balancing product availability, demand variability, service risk, and inventory investment.

In [1]:
# Import required libraries

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

print("Pandas version :", pd.__version__)
print("NumPy version  :", np.__version__)

print(
    "\nInventory Optimization environment initialized successfully."
)

Pandas version : 3.0.6
NumPy version  : 2.5.3

Inventory Optimization environment initialized successfully.


## Step 2 — Define Feature-Engineered Data Path

Define the project directory and the location of the feature-engineered datasets.

The Inventory Optimization analysis will use the existing analysis-ready data to maintain consistency with the previous inventory, demand, and procurement analyses.

In [2]:
# Define project paths

project_path = Path.cwd().parent

featured_data_path = (
    project_path / "data" / "featured"
)

print("Project Path       :", project_path)
print("Featured Data Path :", featured_data_path)

print(
    "\nFeatured Data Folder Exists :",
    featured_data_path.exists()
)

Project Path       : c:\Users\Admin\OneDrive\Documents\Sales_Order_Backlog_Inventory_Analytics
Featured Data Path : c:\Users\Admin\OneDrive\Documents\Sales_Order_Backlog_Inventory_Analytics\data\featured

Featured Data Folder Exists : True


## Step 3 — Load Inventory Optimization Datasets

Load the feature-engineered datasets required for inventory optimization.

The analysis uses:

- **Inventory** — current stock, allocation, safety stock, reorder points, and inventory value
- **Products** — product attributes and cost information
- **Inventory Transactions** — historical demand movements
- **Purchase Orders** — inbound replenishment quantities and procurement status

These datasets will be combined to evaluate inventory shortages, excess stock, replenishment requirements, and optimization opportunities.

In [3]:
# Load Inventory Optimization datasets

inventory = pd.read_csv(
    featured_data_path / "inventory_featured.csv"
)

products = pd.read_csv(
    featured_data_path / "products_featured.csv"
)

inventory_transactions = pd.read_csv(
    featured_data_path / "inventory_transactions_featured.csv",
    parse_dates=["transaction_date"]
)

purchase_orders = pd.read_csv(
    featured_data_path / "purchase_orders_featured.csv",
    parse_dates=[
        "po_date",
        "expected_date",
        "actual_receipt_date"
    ]
)


optimization_datasets = {
    "Inventory": inventory,
    "Products": products,
    "Inventory Transactions": inventory_transactions,
    "Purchase Orders": purchase_orders
}


print("Inventory Optimization datasets loaded successfully.\n")


for name, df in optimization_datasets.items():

    print(
        f"{name:<25} "
        f"{df.shape[0]:>7,} rows × "
        f"{df.shape[1]:>2} columns"
    )

Inventory Optimization datasets loaded successfully.

Inventory                   3,000 rows × 15 columns
Products                      500 rows × 10 columns
Inventory Transactions    110,000 rows ×  6 columns
Purchase Orders             8,500 rows × 23 columns


## Step 4 — Validate Required Fields

Validate that all fields required for inventory optimization are available before performing calculations.

This prevents downstream optimization metrics from being built on missing or incorrectly loaded source columns.

In [4]:
# Define required columns

required_columns = {

    "Inventory": [
        "product_id",
        "warehouse_id",
        "unit_cost",
        "avg_daily_demand",
        "safety_stock",
        "reorder_point",
        "on_hand_qty",
        "allocated_qty",
        "available_qty",
        "inventory_value",
        "stock_status",
        "inventory_days_on_hand",
        "reorder_required_flag"
    ],

    "Products": [
        "product_id",
        "product_name",
        "category",
        "supplier_id",
        "unit_cost"
    ],

    "Inventory Transactions": [
        "transaction_id",
        "transaction_date",
        "product_id",
        "warehouse_id",
        "transaction_type",
        "quantity"
    ],

    "Purchase Orders": [
        "po_id",
        "product_id",
        "warehouse_id",
        "ordered_qty",
        "received_qty",
        "po_status"
    ]
}


validation_results = {}


for name, columns in required_columns.items():

    df = optimization_datasets[name]

    missing_columns = [
        col for col in columns
        if col not in df.columns
    ]

    validation_results[name] = len(missing_columns) == 0

    print(f"\n{name}")
    print("-" * 45)

    print(
        "Required Fields :",
        len(columns)
    )

    print(
        "Missing Fields  :",
        len(missing_columns)
    )

    if missing_columns:
        print("Missing Columns :", missing_columns)
    else:
        print("Status          : PASSED")


all_fields_valid = all(
    validation_results.values()
)


print(
    "\nREQUIRED FIELD VALIDATION :",
    "PASSED"
    if all_fields_valid
    else "REVIEW REQUIRED"
)


Inventory
---------------------------------------------
Required Fields : 13
Missing Fields  : 0
Status          : PASSED

Products
---------------------------------------------
Required Fields : 5
Missing Fields  : 0
Status          : PASSED

Inventory Transactions
---------------------------------------------
Required Fields : 6
Missing Fields  : 0
Status          : PASSED

Purchase Orders
---------------------------------------------
Required Fields : 6
Missing Fields  : 0
Status          : PASSED

REQUIRED FIELD VALIDATION : PASSED


## Step 5 — Build Product–Warehouse Optimization Base

Create the base table for inventory optimization at the **Product–Warehouse** level.

Product attributes are added to the current inventory position so that every inventory record can be analyzed by:

- Product
- Category
- Supplier
- Warehouse
- Current stock
- Available stock
- Safety stock
- Reorder point
- Demand
- Inventory value
- Current stock status

This table will serve as the foundation for subsequent inventory optimization calculations.

In [5]:
# Select product attributes

product_attributes = products[
    [
        "product_id",
        "product_name",
        "category",
        "supplier_id"
    ]
].copy()


# Merge product attributes with inventory

optimization_base = inventory.merge(
    product_attributes,
    on="product_id",
    how="left",
    validate="many_to_one"
)


# Arrange important columns

base_columns = [
    "product_id",
    "product_name",
    "category",
    "supplier_id",
    "warehouse_id",
    "unit_cost",
    "avg_daily_demand",
    "safety_stock",
    "reorder_point",
    "on_hand_qty",
    "allocated_qty",
    "available_qty",
    "inventory_value",
    "inventory_days_on_hand",
    "reorder_required_flag",
    "stock_status"
]


optimization_base = optimization_base[
    base_columns
].copy()


print("Optimization Base Created Successfully\n")

print(
    "Inventory Source Records :",
    f"{len(inventory):,}"
)

print(
    "Optimization Base Records:",
    f"{len(optimization_base):,}"
)

print(
    "Unique Products          :",
    f"{optimization_base['product_id'].nunique():,}"
)

print(
    "Unique Warehouses        :",
    f"{optimization_base['warehouse_id'].nunique():,}"
)

print(
    "Missing Product Matches  :",
    optimization_base["product_name"].isna().sum()
)

print(
    "\nTotal Inventory Value     :",
    f"{optimization_base['inventory_value'].sum():,.2f}"
)

Optimization Base Created Successfully

Inventory Source Records : 3,000
Optimization Base Records: 3,000
Unique Products          : 500
Unique Warehouses        : 6
Missing Product Matches  : 0

Total Inventory Value     : 285,985,182.04


## Step 6 — Calculate Current Inventory Coverage

Calculate inventory coverage at the Product–Warehouse level.

**Inventory Coverage Days** measures how many days the currently available inventory can support expected demand.

Formula:

**Inventory Coverage Days = Available Quantity ÷ Average Daily Demand**

Higher coverage can indicate excess inventory, while very low coverage can indicate replenishment or stockout risk.

In [6]:
# Calculate current inventory coverage

optimization_base["coverage_days"] = np.where(
    optimization_base["avg_daily_demand"] > 0,
    optimization_base["available_qty"]
    / optimization_base["avg_daily_demand"],
    np.nan
)


print("Inventory Coverage Calculated Successfully\n")

print(
    "Average Coverage Days :",
    round(
        optimization_base["coverage_days"].mean(),
        2
    )
)

print(
    "Median Coverage Days  :",
    round(
        optimization_base["coverage_days"].median(),
        2
    )
)

print(
    "Minimum Coverage Days :",
    round(
        optimization_base["coverage_days"].min(),
        2
    )
)

print(
    "Maximum Coverage Days :",
    round(
        optimization_base["coverage_days"].max(),
        2
    )
)

print(
    "Zero Coverage Records :",
    (
        optimization_base["coverage_days"] == 0
    ).sum()
)

print(
    "Missing Coverage      :",
    optimization_base["coverage_days"].isna().sum()
)

Inventory Coverage Calculated Successfully

Average Coverage Days : 32.18
Median Coverage Days  : 31.17
Minimum Coverage Days : 0.0
Maximum Coverage Days : 109.09
Zero Coverage Records : 240
Missing Coverage      : 0


## Step 7 — Create Inventory Coverage Bands

Classify Product–Warehouse inventory records according to the number of days the available stock can support expected demand.

Coverage bands:

- **0 Days** — No available inventory
- **1–7 Days** — Critical coverage
- **8–15 Days** — Low coverage
- **16–30 Days** — Moderate coverage
- **31–60 Days** — Healthy / higher coverage
- **61+ Days** — Potential excess inventory

These bands help identify where replenishment or inventory-reduction actions may be required.

In [7]:
# Create inventory coverage bands

coverage_bins = [
    -0.01,
    0,
    7,
    15,
    30,
    60,
    np.inf
]

coverage_labels = [
    "0 Days",
    "1-7 Days",
    "8-15 Days",
    "16-30 Days",
    "31-60 Days",
    "61+ Days"
]


optimization_base["coverage_band"] = pd.cut(
    optimization_base["coverage_days"],
    bins=coverage_bins,
    labels=coverage_labels
)


coverage_summary = (
    optimization_base
    .groupby(
        "coverage_band",
        observed=False
    )
    .agg(
        inventory_records=("product_id", "size"),
        available_qty=("available_qty", "sum"),
        inventory_value=("inventory_value", "sum")
    )
    .reset_index()
)


coverage_summary["record_share_pct"] = (
    coverage_summary["inventory_records"]
    / len(optimization_base)
    * 100
)


coverage_summary["inventory_value_share_pct"] = (
    coverage_summary["inventory_value"]
    / optimization_base["inventory_value"].sum()
    * 100
)


print("Inventory Coverage Band Summary\n")

print(
    coverage_summary.to_string(
        index=False,
        formatters={
            "inventory_value": "{:,.2f}".format,
            "record_share_pct": "{:.2f}%".format,
            "inventory_value_share_pct": "{:.2f}%".format
        }
    )
)

Inventory Coverage Band Summary

coverage_band  inventory_records  available_qty inventory_value record_share_pct inventory_value_share_pct
       0 Days                240              0      207,396.18            8.00%                     0.07%
     1-7 Days                174            733    4,644,871.75            5.80%                     1.62%
    8-15 Days                327           3096   12,699,803.99           10.90%                     4.44%
   16-30 Days                692          12755   47,789,491.91           23.07%                    16.71%
   31-60 Days               1236          45117  161,574,464.99           41.20%                    56.50%
     61+ Days                331          16518   59,069,153.22           11.03%                    20.65%


## Step 8 — Measure Safety Stock Adequacy

Evaluate whether available inventory is sufficient to maintain the defined safety-stock buffer.

For each Product–Warehouse combination:

**Safety Stock Gap = Available Quantity − Safety Stock**

Interpretation:

- **Above Safety Stock** — sufficient inventory buffer
- **Below Safety Stock** — available inventory has fallen below the desired protection level

This helps identify inventory positions that may require replenishment attention before demand results in further stockout risk.

In [9]:
# Calculate safety stock adequacy

optimization_base["safety_stock_gap"] = (
    optimization_base["available_qty"]
    - optimization_base["safety_stock"]
)


optimization_base["safety_stock_status"] = np.where(
    optimization_base["available_qty"]
    < optimization_base["safety_stock"],
    "Below Safety Stock",
    "At/Above Safety Stock"
)


safety_stock_summary = (
    optimization_base
    .groupby(
        "safety_stock_status"
    )
    .agg(
        inventory_records=("product_id", "size"),
        available_qty=("available_qty", "sum"),
        safety_stock=("safety_stock", "sum"),
        inventory_value=("inventory_value", "sum")
    )
    .reset_index()
)


safety_stock_summary["record_share_pct"] = (
    safety_stock_summary["inventory_records"]
    / len(optimization_base)
    * 100
)


safety_stock_summary["inventory_value_share_pct"] = (
    safety_stock_summary["inventory_value"]
    / optimization_base["inventory_value"].sum()
    * 100
)


print("Safety Stock Adequacy Summary\n")

print(
    safety_stock_summary.to_string(
        index=False,
        formatters={
            "inventory_value": "{:,.2f}".format,
            "record_share_pct": "{:.2f}%".format,
            "inventory_value_share_pct": "{:.2f}%".format
        }
    )
)


print(
    "\nRecords Below Safety Stock :",
    (
        optimization_base["safety_stock_status"]
        == "Below Safety Stock"
    ).sum()
)

Safety Stock Adequacy Summary

  safety_stock_status  inventory_records  available_qty  safety_stock inventory_value record_share_pct inventory_value_share_pct
At/Above Safety Stock               2555          77217         18197  280,255,164.55           85.17%                    98.00%
   Below Safety Stock                445           1002          3430    5,730,017.49           14.83%                     2.00%

Records Below Safety Stock : 445


## Step 9 — Analyze Reorder Point Adequacy

Evaluate current available inventory against the defined reorder point.

A Product–Warehouse record requires replenishment when:

**Available Quantity ≤ Reorder Point**

We also calculate:

**Reorder Gap = Available Quantity − Reorder Point**

A negative or zero reorder gap indicates that the inventory position has reached the replenishment threshold.

This analysis identifies the scale and financial exposure of current replenishment requirements.

In [10]:
# Calculate reorder point adequacy

optimization_base["reorder_gap"] = (
    optimization_base["available_qty"]
    - optimization_base["reorder_point"]
)


optimization_base["reorder_status"] = np.where(
    optimization_base["available_qty"]
    <= optimization_base["reorder_point"],
    "Reorder Required",
    "Above Reorder Point"
)


reorder_summary = (
    optimization_base
    .groupby("reorder_status")
    .agg(
        inventory_records=("product_id", "size"),
        available_qty=("available_qty", "sum"),
        reorder_point=("reorder_point", "sum"),
        inventory_value=("inventory_value", "sum")
    )
    .reset_index()
)


reorder_summary["record_share_pct"] = (
    reorder_summary["inventory_records"]
    / len(optimization_base)
    * 100
)


reorder_summary["inventory_value_share_pct"] = (
    reorder_summary["inventory_value"]
    / optimization_base["inventory_value"].sum()
    * 100
)


print("Reorder Point Adequacy Summary\n")

print(
    reorder_summary.to_string(
        index=False,
        formatters={
            "inventory_value": "{:,.2f}".format,
            "record_share_pct": "{:.2f}%".format,
            "inventory_value_share_pct": "{:.2f}%".format
        }
    )
)


print(
    "\nRecords Requiring Reorder :",
    (
        optimization_base["reorder_status"]
        == "Reorder Required"
    ).sum()
)


# Cross-check against feature-engineered flag

reorder_flag_check = (
    (
        optimization_base["reorder_status"]
        == "Reorder Required"
    )
    ==
    (
        optimization_base["reorder_required_flag"]
        == "Y"
    )
).all()


print(
    "Reorder Flag Reconciliation:",
    "PASSED" if reorder_flag_check else "REVIEW REQUIRED"
)

Reorder Point Adequacy Summary

     reorder_status  inventory_records  available_qty  reorder_point inventory_value record_share_pct inventory_value_share_pct
Above Reorder Point               1689          65295          37595  234,506,268.15           56.30%                    82.00%
   Reorder Required               1311          12924          27712   51,478,913.89           43.70%                    18.00%

Records Requiring Reorder : 1311
Reorder Flag Reconciliation: PASSED


## Step 10 — Calculate Replenishment Quantity to Reorder Point

Estimate the immediate replenishment quantity required to restore inventory to the defined reorder point.

For records where available inventory is below the reorder point:

**Replenishment Gap Quantity = Reorder Point − Available Quantity**

For records already above the reorder point, the replenishment gap is zero.

This metric quantifies the current inventory shortfall across Product–Warehouse combinations.

In [11]:
# Calculate replenishment gap quantity

optimization_base["replenishment_gap_qty"] = np.maximum(
    optimization_base["reorder_point"]
    - optimization_base["available_qty"],
    0
)


# Estimate value of replenishment gap

optimization_base["replenishment_gap_value"] = (
    optimization_base["replenishment_gap_qty"]
    * optimization_base["unit_cost"]
)


replenishment_required = optimization_base[
    optimization_base["replenishment_gap_qty"] > 0
].copy()


print("Replenishment Gap Analysis\n")

print(
    "Records with Positive Replenishment Gap :",
    f"{len(replenishment_required):,}"
)

print(
    "Total Replenishment Gap Quantity        :",
    f"{replenishment_required['replenishment_gap_qty'].sum():,.0f}"
)

print(
    "Estimated Replenishment Gap Value       :",
    f"{replenishment_required['replenishment_gap_value'].sum():,.2f}"
)

print(
    "Average Gap Quantity per Record         :",
    f"{replenishment_required['replenishment_gap_qty'].mean():,.2f}"
)

print(
    "Maximum Gap Quantity                    :",
    f"{replenishment_required['replenishment_gap_qty'].max():,.0f}"
)

Replenishment Gap Analysis

Records with Positive Replenishment Gap : 1,227
Total Replenishment Gap Quantity        : 14,788
Estimated Replenishment Gap Value       : 48,258,355.89
Average Gap Quantity per Record         : 12.05
Maximum Gap Quantity                    : 105


## Step 11 — Account for Open Purchase Orders

Calculate outstanding inbound inventory from purchase orders.

For each Purchase Order:

**Outstanding PO Quantity = Ordered Quantity − Received Quantity**

Only positive outstanding quantities represent inventory that is still expected to arrive.

Outstanding quantities are then aggregated at the Product–Warehouse level and merged with the optimization base.

This prevents the optimization model from recommending additional replenishment when sufficient inventory is already on order.

In [12]:
# Calculate outstanding purchase order quantity

purchase_orders["outstanding_po_qty"] = np.maximum(
    purchase_orders["ordered_qty"]
    - purchase_orders["received_qty"],
    0
)


# Aggregate outstanding inbound quantity by product and warehouse

open_po_summary = (
    purchase_orders[
        purchase_orders["outstanding_po_qty"] > 0
    ]
    .groupby(
        ["product_id", "warehouse_id"],
        as_index=False
    )
    .agg(
        outstanding_po_qty=("outstanding_po_qty", "sum"),
        open_po_count=("po_id", "nunique")
    )
)


# Merge inbound inventory into optimization base

optimization_base = optimization_base.merge(
    open_po_summary,
    on=["product_id", "warehouse_id"],
    how="left",
    validate="one_to_one"
)


optimization_base[
    ["outstanding_po_qty", "open_po_count"]
] = optimization_base[
    ["outstanding_po_qty", "open_po_count"]
].fillna(0)


print("Outstanding Purchase Order Analysis\n")

print(
    "Product-Warehouse Records with Inbound Stock :",
    f"{(optimization_base['outstanding_po_qty'] > 0).sum():,}"
)

print(
    "Total Outstanding PO Quantity                :",
    f"{optimization_base['outstanding_po_qty'].sum():,.0f}"
)

print(
    "Total POs with Outstanding Quantity          :",
    f"{purchase_orders.loc[purchase_orders['outstanding_po_qty'] > 0, 'po_id'].nunique():,}"
)

print(
    "Maximum Outstanding Qty per Product-Warehouse:",
    f"{optimization_base['outstanding_po_qty'].max():,.0f}"
)

print(
    "Missing Inbound Quantity                    :",
    optimization_base["outstanding_po_qty"].isna().sum()
)

Outstanding Purchase Order Analysis

Product-Warehouse Records with Inbound Stock : 2,525
Total Outstanding PO Quantity                : 132,826
Total POs with Outstanding Quantity          : 8,500
Maximum Outstanding Qty per Product-Warehouse: 576
Missing Inbound Quantity                    : 0


## Step 12 — Calculate Net Replenishment Requirement After Inbound Stock

Adjust the replenishment requirement for inventory that is already expected through outstanding purchase orders.

Formula:

**Projected Available Quantity = Available Quantity + Outstanding PO Quantity**

and:

**Net Replenishment Requirement = MAX(Reorder Point − Projected Available Quantity, 0)**

This distinguishes between:

- Shortages already covered by inbound purchase orders
- Shortages only partially covered by inbound inventory
- Remaining replenishment requirements after considering inbound supply

This provides a more realistic replenishment recommendation than using current available inventory alone.

In [13]:
# Calculate projected inventory after inbound purchase orders

optimization_base["projected_available_qty"] = (
    optimization_base["available_qty"]
    + optimization_base["outstanding_po_qty"]
)


# Calculate net replenishment requirement

optimization_base["net_replenishment_qty"] = np.maximum(
    optimization_base["reorder_point"]
    - optimization_base["projected_available_qty"],
    0
)


# Estimate value of remaining replenishment requirement

optimization_base["net_replenishment_value"] = (
    optimization_base["net_replenishment_qty"]
    * optimization_base["unit_cost"]
)


# Identify records with remaining shortages

net_replenishment_required = optimization_base[
    optimization_base["net_replenishment_qty"] > 0
].copy()


print("Net Replenishment Requirement After Inbound Stock\n")

print(
    "Original Replenishment Gap Quantity :",
    f"{optimization_base['replenishment_gap_qty'].sum():,.0f}"
)

print(
    "Outstanding Inbound PO Quantity     :",
    f"{optimization_base['outstanding_po_qty'].sum():,.0f}"
)

print(
    "Remaining Net Replenishment Quantity:",
    f"{optimization_base['net_replenishment_qty'].sum():,.0f}"
)

print(
    "Records Still Requiring Replenishment:",
    f"{len(net_replenishment_required):,}"
)

print(
    "Remaining Replenishment Value       :",
    f"{optimization_base['net_replenishment_value'].sum():,.2f}"
)


covered_by_inbound = (
    (optimization_base["replenishment_gap_qty"] > 0)
    &
    (optimization_base["net_replenishment_qty"] == 0)
).sum()


print(
    "Shortage Records Fully Covered by Inbound:",
    f"{covered_by_inbound:,}"
)

Net Replenishment Requirement After Inbound Stock

Original Replenishment Gap Quantity : 14,788
Outstanding Inbound PO Quantity     : 132,826
Remaining Net Replenishment Quantity: 1,984
Records Still Requiring Replenishment: 309
Remaining Replenishment Value       : 6,850,078.47
Shortage Records Fully Covered by Inbound: 918


## Step 13 — Identify Excess Inventory Quantity

Estimate potential excess inventory using a 60-day demand coverage benchmark.

For each Product–Warehouse combination:

**60-Day Target Inventory = Average Daily Demand × 60**

Potential excess quantity is calculated as:

**Excess Inventory Quantity = MAX(Available Quantity − 60-Day Target Inventory, 0)**

This identifies inventory above approximately two months of expected demand.

The calculation is an optimization screening measure rather than an automatic disposal recommendation. Product importance, demand variability, supplier lead time, and ABC–XYZ classification will be considered before final recommendations are made.

In [14]:
# Calculate 60-day target inventory

optimization_base["target_60_day_qty"] = (
    optimization_base["avg_daily_demand"] * 60
)


# Calculate potential excess inventory quantity

optimization_base["excess_inventory_qty"] = np.maximum(
    optimization_base["available_qty"]
    - optimization_base["target_60_day_qty"],
    0
)


# Estimate potential excess inventory value

optimization_base["excess_inventory_value"] = (
    optimization_base["excess_inventory_qty"]
    * optimization_base["unit_cost"]
)


excess_inventory = optimization_base[
    optimization_base["excess_inventory_qty"] > 0
].copy()


print("Potential Excess Inventory Analysis\n")

print(
    "Records with Potential Excess Inventory :",
    f"{len(excess_inventory):,}"
)

print(
    "Potential Excess Inventory Quantity      :",
    f"{excess_inventory['excess_inventory_qty'].sum():,.2f}"
)

print(
    "Potential Excess Inventory Value         :",
    f"{excess_inventory['excess_inventory_value'].sum():,.2f}"
)

print(
    "Share of Total Inventory Value           :",
    f"{(
        excess_inventory['excess_inventory_value'].sum()
        / optimization_base['inventory_value'].sum()
        * 100
    ):.2f}%"
)

print(
    "Average Excess Quantity per Record       :",
    f"{excess_inventory['excess_inventory_qty'].mean():,.2f}"
)

Potential Excess Inventory Analysis

Records with Potential Excess Inventory : 330
Potential Excess Inventory Quantity      : 1,903.80
Potential Excess Inventory Value         : 6,631,536.78
Share of Total Inventory Value           : 2.32%
Average Excess Quantity per Record       : 5.77


## Step 14 — Identify Inventory Optimization Position

Classify each Product–Warehouse inventory position according to its current optimization requirement.

The classification uses the replenishment requirement after considering inbound purchase orders and the 60-day excess-inventory benchmark.

Inventory positions are classified as:

- **Replenishment Required** — additional inventory is still required after considering inbound supply
- **Potential Excess** — available inventory exceeds the 60-day demand benchmark
- **Balanced / Monitor** — neither a remaining replenishment shortage nor potential excess is identified

This creates a simple decision-oriented view of the current inventory portfolio.

In [15]:
# Create inventory optimization status

conditions = [
    optimization_base["net_replenishment_qty"] > 0,
    optimization_base["excess_inventory_qty"] > 0
]

choices = [
    "Replenishment Required",
    "Potential Excess"
]


optimization_base["optimization_status"] = np.select(
    conditions,
    choices,
    default="Balanced / Monitor"
)


optimization_status_summary = (
    optimization_base
    .groupby("optimization_status")
    .agg(
        inventory_records=("product_id", "size"),
        unique_products=("product_id", "nunique"),
        available_qty=("available_qty", "sum"),
        inventory_value=("inventory_value", "sum"),
        net_replenishment_qty=("net_replenishment_qty", "sum"),
        excess_inventory_qty=("excess_inventory_qty", "sum"),
        excess_inventory_value=("excess_inventory_value", "sum")
    )
    .reset_index()
)


optimization_status_summary["record_share_pct"] = (
    optimization_status_summary["inventory_records"]
    / len(optimization_base)
    * 100
)


optimization_status_summary["inventory_value_share_pct"] = (
    optimization_status_summary["inventory_value"]
    / optimization_base["inventory_value"].sum()
    * 100
)


print("Inventory Optimization Position Summary\n")

print(
    optimization_status_summary.to_string(
        index=False,
        formatters={
            "inventory_value": "{:,.2f}".format,
            "net_replenishment_qty": "{:,.2f}".format,
            "excess_inventory_qty": "{:,.2f}".format,
            "excess_inventory_value": "{:,.2f}".format,
            "record_share_pct": "{:.2f}%".format,
            "inventory_value_share_pct": "{:.2f}%".format
        }
    )
)


print(
    "\nTotal Records :",
    optimization_status_summary[
        "inventory_records"
    ].sum()
)

Inventory Optimization Position Summary

   optimization_status  inventory_records  unique_products  available_qty inventory_value net_replenishment_qty excess_inventory_qty excess_inventory_value record_share_pct inventory_value_share_pct
    Balanced / Monitor               2361              500          60120  220,189,622.77                  0.00                 0.00                   0.00           78.70%                    76.99%
      Potential Excess                330              257          16449   58,621,083.96                  0.00             1,903.80           6,631,536.78           11.00%                    20.50%
Replenishment Required                309              215           1650    7,174,475.31              1,984.00                 0.00                   0.00           10.30%                     2.51%

Total Records : 3000


## Step 15 — Integrate ABC–XYZ Segmentation

Integrate the validated ABC–XYZ segmentation into the Product–Warehouse optimization model.

ABC classification represents the financial importance of inventory, while XYZ classification represents historical demand variability.

Combining ABC–XYZ segmentation with the current optimization position allows replenishment and excess-stock decisions to be prioritized according to both:

- Inventory investment importance
- Demand predictability
- Current stock position

The classification methodology remains consistent with the completed ABC–XYZ Analysis notebook.

In [16]:
# ---------------------------------------------------------
# Recreate validated ABC classification
# ---------------------------------------------------------

product_inventory_value = (
    inventory
    .groupby("product_id", as_index=False)
    .agg(
        total_inventory_value=("inventory_value", "sum")
    )
    .sort_values(
        "total_inventory_value",
        ascending=False
    )
)


product_inventory_value["inventory_value_share"] = (
    product_inventory_value["total_inventory_value"]
    / product_inventory_value["total_inventory_value"].sum()
)


product_inventory_value["cumulative_inventory_share"] = (
    product_inventory_value["inventory_value_share"].cumsum()
)


product_inventory_value["abc_class"] = np.select(
    [
        product_inventory_value[
            "cumulative_inventory_share"
        ] <= 0.80,

        product_inventory_value[
            "cumulative_inventory_share"
        ] <= 0.95
    ],
    [
        "A",
        "B"
    ],
    default="C"
)


# ---------------------------------------------------------
# Recreate validated XYZ classification
# ---------------------------------------------------------

sales_issue = inventory_transactions[
    inventory_transactions["transaction_type"]
    == "SALE_ISSUE"
].copy()


sales_issue["demand_qty"] = (
    sales_issue["quantity"].abs()
)


sales_issue["year_month"] = (
    sales_issue["transaction_date"]
    .dt.to_period("M")
)


monthly_demand = (
    sales_issue
    .groupby(
        ["product_id", "year_month"],
        as_index=False
    )
    .agg(
        monthly_demand=("demand_qty", "sum")
    )
)


all_months = pd.period_range(
    sales_issue["transaction_date"].min().to_period("M"),
    sales_issue["transaction_date"].max().to_period("M"),
    freq="M"
)


complete_index = pd.MultiIndex.from_product(
    [
        products["product_id"].unique(),
        all_months
    ],
    names=[
        "product_id",
        "year_month"
    ]
)


complete_monthly_demand = (
    monthly_demand
    .set_index(
        ["product_id", "year_month"]
    )
    .reindex(
        complete_index,
        fill_value=0
    )
    .reset_index()
)


xyz_metrics = (
    complete_monthly_demand
    .groupby(
        "product_id",
        as_index=False
    )
    .agg(
        avg_monthly_demand=(
            "monthly_demand",
            "mean"
        ),
        demand_std=(
            "monthly_demand",
            "std"
        )
    )
)


xyz_metrics["demand_cv"] = (
    xyz_metrics["demand_std"]
    / xyz_metrics["avg_monthly_demand"]
)


xyz_metrics["xyz_class"] = np.select(
    [
        xyz_metrics["demand_cv"] <= 0.50,
        xyz_metrics["demand_cv"] <= 1.00
    ],
    [
        "X",
        "Y"
    ],
    default="Z"
)


# ---------------------------------------------------------
# Create ABC-XYZ mapping
# ---------------------------------------------------------

abc_xyz_mapping = (
    product_inventory_value[
        [
            "product_id",
            "abc_class"
        ]
    ]
    .merge(
        xyz_metrics[
            [
                "product_id",
                "xyz_class",
                "demand_cv"
            ]
        ],
        on="product_id",
        how="inner",
        validate="one_to_one"
    )
)


abc_xyz_mapping["abc_xyz_segment"] = (
    abc_xyz_mapping["abc_class"]
    + abc_xyz_mapping["xyz_class"]
)


# Merge into optimization base

optimization_base = optimization_base.merge(
    abc_xyz_mapping,
    on="product_id",
    how="left",
    validate="many_to_one"
)


print("ABC-XYZ Integration Completed\n")

print(
    optimization_base[
        "abc_xyz_segment"
    ].value_counts().sort_index()
)

print(
    "\nMissing ABC-XYZ Classifications :",
    optimization_base[
        "abc_xyz_segment"
    ].isna().sum()
)

print(
    "Unique Classified Products     :",
    abc_xyz_mapping[
        "product_id"
    ].nunique()
)

ABC-XYZ Integration Completed

abc_xyz_segment
AX    480
AY    918
AZ     18
BX    144
BY    540
BZ    108
CX     72
CY    408
CZ    312
Name: count, dtype: int64

Missing ABC-XYZ Classifications : 0
Unique Classified Products     : 500


## Step 16 — Create ABC–XYZ-Based Replenishment Priority

Prioritize remaining replenishment requirements using ABC–XYZ segmentation.

Priority logic:

- **Critical Priority** — AX, AY, AZ products requiring replenishment
- **High Priority** — BX, BY, BZ products requiring replenishment
- **Moderate Priority** — CX, CY products requiring replenishment
- **Low Priority** — CZ products requiring replenishment
- **No Immediate Replenishment** — records with no remaining replenishment requirement

This ensures that inventory investment and demand behavior are considered when deciding which shortages should be addressed first.

In [17]:
# Define replenishment priority

def assign_replenishment_priority(row):

    if row["net_replenishment_qty"] <= 0:
        return "No Immediate Replenishment"

    segment = row["abc_xyz_segment"]

    if segment in ["AX", "AY", "AZ"]:
        return "Critical Priority"

    elif segment in ["BX", "BY", "BZ"]:
        return "High Priority"

    elif segment in ["CX", "CY"]:
        return "Moderate Priority"

    else:
        return "Low Priority"


optimization_base["replenishment_priority"] = (
    optimization_base.apply(
        assign_replenishment_priority,
        axis=1
    )
)


priority_order = [
    "Critical Priority",
    "High Priority",
    "Moderate Priority",
    "Low Priority"
]


replenishment_priority_summary = (
    optimization_base[
        optimization_base["net_replenishment_qty"] > 0
    ]
    .groupby(
        "replenishment_priority"
    )
    .agg(
        inventory_records=("product_id", "size"),
        unique_products=("product_id", "nunique"),
        net_replenishment_qty=("net_replenishment_qty", "sum"),
        net_replenishment_value=("net_replenishment_value", "sum")
    )
    .reindex(priority_order)
    .fillna(0)
    .reset_index()
)


replenishment_priority_summary[
    "replenishment_value_share_pct"
] = (
    replenishment_priority_summary[
        "net_replenishment_value"
    ]
    / optimization_base[
        "net_replenishment_value"
    ].sum()
    * 100
)


print("ABC-XYZ Replenishment Priority Summary\n")

print(
    replenishment_priority_summary.to_string(
        index=False,
        formatters={
            "net_replenishment_qty": "{:,.2f}".format,
            "net_replenishment_value": "{:,.2f}".format,
            "replenishment_value_share_pct": "{:.2f}%".format
        }
    )
)


print(
    "\nTotal Prioritized Replenishment Records :",
    replenishment_priority_summary[
        "inventory_records"
    ].sum()
)

ABC-XYZ Replenishment Priority Summary

replenishment_priority  inventory_records  unique_products net_replenishment_qty net_replenishment_value replenishment_value_share_pct
     Critical Priority                 97               80                834.00            3,792,088.79                        55.36%
         High Priority                 75               53                501.00            1,650,180.46                        24.09%
     Moderate Priority                 62               40                413.00              803,879.57                        11.74%
          Low Priority                 75               42                236.00              603,929.65                         8.82%

Total Prioritized Replenishment Records : 309


## Step 17 — Create ABC–XYZ-Based Excess Inventory Priority

Prioritize potential excess inventory using ABC–XYZ segmentation.

Reduction priority logic:

- **Highest Reduction Priority** — CZ and BZ
- **High Reduction Priority** — CY and AZ
- **Moderate Reduction Priority** — BY and CX
- **Low Reduction Priority** — AY and BX
- **Protect / Review Carefully** — AX

This approach prevents the business from treating all excess inventory equally.

Stable, high-value AX products should be reviewed cautiously, while low-value or highly unpredictable inventory can receive stronger reduction attention.

In [18]:
# Define excess inventory reduction priority

def assign_excess_priority(row):

    if row["excess_inventory_qty"] <= 0:
        return "No Excess Inventory"

    segment = row["abc_xyz_segment"]

    if segment in ["CZ", "BZ"]:
        return "Highest Reduction Priority"

    elif segment in ["CY", "AZ"]:
        return "High Reduction Priority"

    elif segment in ["BY", "CX"]:
        return "Moderate Reduction Priority"

    elif segment in ["AY", "BX"]:
        return "Low Reduction Priority"

    else:
        return "Protect / Review Carefully"


optimization_base["excess_reduction_priority"] = (
    optimization_base.apply(
        assign_excess_priority,
        axis=1
    )
)


excess_priority_order = [
    "Highest Reduction Priority",
    "High Reduction Priority",
    "Moderate Reduction Priority",
    "Low Reduction Priority",
    "Protect / Review Carefully"
]


excess_priority_summary = (
    optimization_base[
        optimization_base["excess_inventory_qty"] > 0
    ]
    .groupby(
        "excess_reduction_priority"
    )
    .agg(
        inventory_records=("product_id", "size"),
        unique_products=("product_id", "nunique"),
        excess_inventory_qty=("excess_inventory_qty", "sum"),
        excess_inventory_value=("excess_inventory_value", "sum")
    )
    .reindex(excess_priority_order)
    .fillna(0)
    .reset_index()
)


excess_priority_summary[
    "excess_value_share_pct"
] = (
    excess_priority_summary[
        "excess_inventory_value"
    ]
    / optimization_base[
        "excess_inventory_value"
    ].sum()
    * 100
)


print("ABC-XYZ Excess Inventory Priority Summary\n")

print(
    excess_priority_summary.to_string(
        index=False,
        formatters={
            "excess_inventory_qty": "{:,.2f}".format,
            "excess_inventory_value": "{:,.2f}".format,
            "excess_value_share_pct": "{:.2f}%".format
        }
    )
)


print(
    "\nTotal Prioritized Excess Records :",
    excess_priority_summary[
        "inventory_records"
    ].sum()
)

ABC-XYZ Excess Inventory Priority Summary

  excess_reduction_priority  inventory_records  unique_products excess_inventory_qty excess_inventory_value excess_value_share_pct
 Highest Reduction Priority                 66               44               119.40             367,065.14                  5.54%
    High Reduction Priority                 38               34               180.20             243,928.98                  3.68%
Moderate Reduction Priority                 65               48               323.60             737,073.51                 11.11%
     Low Reduction Priority                116               93               743.60           3,047,010.15                 45.95%
 Protect / Review Carefully                 45               38               537.00           2,236,459.00                 33.72%

Total Prioritized Excess Records : 330


## Step 18 — Identify Top Critical Replenishment Records

Identify the highest-value replenishment requirements among Critical Priority ABC–XYZ products.

These records represent the most important immediate replenishment opportunities because they combine:

- Remaining inventory shortage after considering inbound purchase orders
- High financial importance
- ABC–XYZ-based criticality

Ranking these records helps procurement and inventory teams focus first on shortages with the greatest business exposure.

In [19]:
# Identify critical replenishment records

critical_replenishment = (
    optimization_base[
        optimization_base["replenishment_priority"]
        == "Critical Priority"
    ]
    .sort_values(
        "net_replenishment_value",
        ascending=False
    )
    .copy()
)


top_critical_replenishment = (
    critical_replenishment[
        [
            "product_id",
            "product_name",
            "category",
            "warehouse_id",
            "abc_xyz_segment",
            "available_qty",
            "reorder_point",
            "outstanding_po_qty",
            "projected_available_qty",
            "net_replenishment_qty",
            "net_replenishment_value"
        ]
    ]
    .head(10)
)


print("Top 10 Critical Replenishment Records\n")

print(
    top_critical_replenishment.to_string(
        index=False,
        formatters={
            "net_replenishment_value": "{:,.2f}".format
        }
    )
)


print(
    "\nTotal Critical Replenishment Records :",
    f"{len(critical_replenishment):,}"
)

print(
    "Critical Replenishment Quantity      :",
    f"{critical_replenishment['net_replenishment_qty'].sum():,.0f}"
)

print(
    "Critical Replenishment Value         :",
    f"{critical_replenishment['net_replenishment_value'].sum():,.2f}"
)

Top 10 Critical Replenishment Records

product_id             product_name               category warehouse_id abc_xyz_segment  available_qty  reorder_point  outstanding_po_qty  projected_available_qty  net_replenishment_qty net_replenishment_value
   SKU0021    Wellness Product 0021 Health & Personal Care         WH06              AX              0             40                 9.0                      9.0                   31.0              195,705.48
   SKU0182       Audio Product 0182            Electronics         WH01              AX              5             36                 0.0                      5.0                   31.0              124,324.26
   SKU0331  Stationery Product 0331        Office Supplies         WH04              AY              0             28                 8.0                      8.0                   20.0              115,863.00
   SKU0463 Accessories Product 0463            Electronics         WH06              AY              6             27    

## Step 19 — Identify Top Excess-Inventory Reduction Opportunities

Identify the strongest potential inventory-reduction opportunities.

The analysis focuses on Product–Warehouse records classified as:

- Highest Reduction Priority
- High Reduction Priority

These records combine excess inventory with ABC–XYZ characteristics that make inventory reduction more appropriate.

The results provide a targeted working-capital optimization list rather than assuming that all inventory above the 60-day benchmark should automatically be reduced.

In [20]:
# Identify strongest excess inventory reduction opportunities

reduction_opportunities = (
    optimization_base[
        optimization_base[
            "excess_reduction_priority"
        ].isin(
            [
                "Highest Reduction Priority",
                "High Reduction Priority"
            ]
        )
    ]
    .sort_values(
        "excess_inventory_value",
        ascending=False
    )
    .copy()
)


top_reduction_opportunities = (
    reduction_opportunities[
        [
            "product_id",
            "product_name",
            "category",
            "warehouse_id",
            "abc_xyz_segment",
            "available_qty",
            "coverage_days",
            "target_60_day_qty",
            "excess_inventory_qty",
            "excess_inventory_value",
            "excess_reduction_priority"
        ]
    ]
    .head(10)
)


print("Top 10 Excess Inventory Reduction Opportunities\n")

print(
    top_reduction_opportunities.to_string(
        index=False,
        formatters={
            "coverage_days": "{:.2f}".format,
            "target_60_day_qty": "{:.2f}".format,
            "excess_inventory_qty": "{:.2f}".format,
            "excess_inventory_value": "{:,.2f}".format
        }
    )
)


print(
    "\nTargeted Reduction Records :",
    f"{len(reduction_opportunities):,}"
)

print(
    "Targeted Excess Quantity  :",
    f"{reduction_opportunities['excess_inventory_qty'].sum():,.2f}"
)

print(
    "Targeted Excess Value     :",
    f"{reduction_opportunities['excess_inventory_value'].sum():,.2f}"
)

print(
    "Share of Total Potential Excess Value :",
    f"{(
        reduction_opportunities['excess_inventory_value'].sum()
        / optimization_base['excess_inventory_value'].sum()
        * 100
    ):.2f}%"
)

Top 10 Excess Inventory Reduction Opportunities

product_id               product_name               category warehouse_id abc_xyz_segment  available_qty coverage_days target_60_day_qty excess_inventory_qty excess_inventory_value  excess_reduction_priority
   SKU0035 Smart Devices Product 0035            Electronics         WH06              CZ             12        109.09              6.60                 5.40              32,561.24 Highest Reduction Priority
   SKU0175    Stationery Product 0175        Office Supplies         WH04              CY             30         88.24             20.40                 9.60              23,054.11    High Reduction Priority
   SKU0426      Cookware Product 0426         Home & Kitchen         WH01              AZ             27         69.23             23.40                 3.60              21,978.86    High Reduction Priority
   SKU0219    Stationery Product 0219        Office Supplies         WH06              CZ             13         92.86 

## Step 20 — Analyze Inventory Optimization by Warehouse

Evaluate inventory optimization opportunities across warehouses.

For each warehouse, measure:

- Inventory records
- Current inventory value
- Remaining replenishment quantity
- Remaining replenishment value
- Potential excess quantity
- Potential excess value
- Number of replenishment-required records
- Number of potential-excess records

This identifies warehouses where inventory imbalance is most concentrated and supports warehouse-level inventory planning.

In [21]:
# Analyze inventory optimization by warehouse

warehouse_optimization = (
    optimization_base
    .groupby(
        "warehouse_id",
        as_index=False
    )
    .agg(
        inventory_records=("product_id", "size"),
        inventory_value=("inventory_value", "sum"),
        net_replenishment_qty=("net_replenishment_qty", "sum"),
        net_replenishment_value=("net_replenishment_value", "sum"),
        excess_inventory_qty=("excess_inventory_qty", "sum"),
        excess_inventory_value=("excess_inventory_value", "sum")
    )
)


# Count optimization statuses by warehouse

warehouse_status_counts = (
    optimization_base
    .pivot_table(
        index="warehouse_id",
        columns="optimization_status",
        values="product_id",
        aggfunc="size",
        fill_value=0
    )
    .reset_index()
)


warehouse_optimization = (
    warehouse_optimization
    .merge(
        warehouse_status_counts,
        on="warehouse_id",
        how="left"
    )
)


warehouse_optimization = (
    warehouse_optimization
    .sort_values(
        "net_replenishment_value",
        ascending=False
    )
)


print("Warehouse Inventory Optimization Summary\n")

print(
    warehouse_optimization.to_string(
        index=False,
        formatters={
            "inventory_value": "{:,.2f}".format,
            "net_replenishment_qty": "{:,.2f}".format,
            "net_replenishment_value": "{:,.2f}".format,
            "excess_inventory_qty": "{:,.2f}".format,
            "excess_inventory_value": "{:,.2f}".format
        }
    )
)

Warehouse Inventory Optimization Summary

warehouse_id  inventory_records inventory_value net_replenishment_qty net_replenishment_value excess_inventory_qty excess_inventory_value  Balanced / Monitor  Potential Excess  Replenishment Required
        WH06                500   45,358,031.66                446.00            1,769,724.53               264.20             916,604.27                 392                54                      54
        WH01                500   48,514,594.37                394.00            1,269,808.80               424.20           1,504,644.68                 378                57                      65
        WH02                500   47,404,063.86                311.00            1,118,668.06               235.60             752,278.73                 396                53                      51
        WH04                500   47,126,003.30                321.00            1,039,438.00               351.00           1,407,585.85                 407 

## Step 21 — Analyze Inventory Optimization by Category

Evaluate inventory optimization opportunities across product categories.

For each category, measure:

- Number of Product–Warehouse records
- Current inventory value
- Remaining replenishment quantity and value
- Potential excess quantity and value
- Replenishment-required records
- Potential-excess records

This identifies the categories where inventory imbalance creates the greatest replenishment requirement or working-capital optimization opportunity.

In [22]:
# Analyze inventory optimization by category

category_optimization = (
    optimization_base
    .groupby(
        "category",
        as_index=False
    )
    .agg(
        inventory_records=("product_id", "size"),
        unique_products=("product_id", "nunique"),
        inventory_value=("inventory_value", "sum"),
        net_replenishment_qty=("net_replenishment_qty", "sum"),
        net_replenishment_value=("net_replenishment_value", "sum"),
        excess_inventory_qty=("excess_inventory_qty", "sum"),
        excess_inventory_value=("excess_inventory_value", "sum")
    )
)


# Count optimization statuses by category

category_status_counts = (
    optimization_base
    .pivot_table(
        index="category",
        columns="optimization_status",
        values="product_id",
        aggfunc="size",
        fill_value=0
    )
    .reset_index()
)


category_optimization = (
    category_optimization
    .merge(
        category_status_counts,
        on="category",
        how="left"
    )
)


category_optimization = (
    category_optimization
    .sort_values(
        "net_replenishment_value",
        ascending=False
    )
)


print("Category Inventory Optimization Summary\n")

print(
    category_optimization.to_string(
        index=False,
        formatters={
            "inventory_value": "{:,.2f}".format,
            "net_replenishment_qty": "{:,.2f}".format,
            "net_replenishment_value": "{:,.2f}".format,
            "excess_inventory_qty": "{:,.2f}".format,
            "excess_inventory_value": "{:,.2f}".format
        }
    )
)

Category Inventory Optimization Summary

              category  inventory_records  unique_products inventory_value net_replenishment_qty net_replenishment_value excess_inventory_qty excess_inventory_value  Balanced / Monitor  Potential Excess  Replenishment Required
           Electronics                732              122   79,180,029.99                575.00            2,153,591.52               426.40           1,561,452.42                 591                72                      69
Health & Personal Care                576               96   50,175,615.88                379.00            1,344,713.57               392.80           1,138,873.71                 446                68                      62
       Office Supplies                582               97   55,462,605.80                402.00            1,341,518.56               383.00           1,281,828.56                 453                65                      64
        Home & Kitchen                654          

## Step 22 — Create Final Recommended Inventory Action

Translate the inventory optimization results into actionable Product–Warehouse recommendations.

Recommended actions combine:

- Remaining replenishment requirement
- ABC–XYZ replenishment priority
- Potential excess inventory
- ABC–XYZ excess-reduction priority
- Balanced inventory positions

The resulting action field provides a practical decision layer that can later be used in SQL analysis, dashboards, and management reporting.

In [23]:
# Create final recommended inventory action

def assign_inventory_action(row):

    # Replenishment actions
    if row["net_replenishment_qty"] > 0:

        if row["replenishment_priority"] == "Critical Priority":
            return "Urgent Replenishment"

        elif row["replenishment_priority"] == "High Priority":
            return "High-Priority Replenishment"

        elif row["replenishment_priority"] == "Moderate Priority":
            return "Planned Replenishment"

        else:
            return "Low-Priority Replenishment"


    # Excess inventory actions
    if row["excess_inventory_qty"] > 0:

        if row["excess_reduction_priority"] == "Highest Reduction Priority":
            return "Reduce / Rationalize Stock"

        elif row["excess_reduction_priority"] == "High Reduction Priority":
            return "Reduce Excess Stock"

        elif row["excess_reduction_priority"] == "Moderate Reduction Priority":
            return "Review Excess Stock"

        elif row["excess_reduction_priority"] == "Low Reduction Priority":
            return "Monitor Excess Stock"

        else:
            return "Protect / Review Excess"


    # Balanced inventory
    return "Maintain / Monitor"


optimization_base["recommended_action"] = (
    optimization_base.apply(
        assign_inventory_action,
        axis=1
    )
)


action_summary = (
    optimization_base
    .groupby(
        "recommended_action"
    )
    .agg(
        inventory_records=("product_id", "size"),
        unique_products=("product_id", "nunique"),
        inventory_value=("inventory_value", "sum"),
        net_replenishment_qty=("net_replenishment_qty", "sum"),
        net_replenishment_value=("net_replenishment_value", "sum"),
        excess_inventory_qty=("excess_inventory_qty", "sum"),
        excess_inventory_value=("excess_inventory_value", "sum")
    )
    .reset_index()
    .sort_values(
        "inventory_records",
        ascending=False
    )
)


print("Recommended Inventory Action Summary\n")

print(
    action_summary.to_string(
        index=False,
        formatters={
            "inventory_value": "{:,.2f}".format,
            "net_replenishment_qty": "{:,.2f}".format,
            "net_replenishment_value": "{:,.2f}".format,
            "excess_inventory_qty": "{:,.2f}".format,
            "excess_inventory_value": "{:,.2f}".format
        }
    )
)


print(
    "\nTotal Classified Records :",
    action_summary[
        "inventory_records"
    ].sum()
)

print(
    "Missing Recommended Actions :",
    optimization_base[
        "recommended_action"
    ].isna().sum()
)

Recommended Inventory Action Summary

         recommended_action  inventory_records  unique_products inventory_value net_replenishment_qty net_replenishment_value excess_inventory_qty excess_inventory_value
         Maintain / Monitor               2361              500  220,189,622.77                  0.00                    0.00                 0.00                   0.00
       Monitor Excess Stock                116               93   27,023,013.75                  0.00                    0.00               743.60           3,047,010.15
       Urgent Replenishment                 97               80    4,490,821.63                834.00            3,792,088.79                 0.00                   0.00
High-Priority Replenishment                 75               53    1,571,030.88                501.00            1,650,180.46                 0.00                   0.00
 Low-Priority Replenishment                 75               42      470,325.55                236.00           

## Step 23 — Inventory Optimization Reconciliation

Reconcile the major inventory optimization calculations before drawing final business conclusions.

The reconciliation verifies:

- Product–Warehouse record count
- Total inventory value
- Replenishment gap quantity
- Outstanding purchase-order quantity
- Net replenishment quantity and value
- Potential excess quantity and value
- Optimization-status classification
- ABC–XYZ integration
- Replenishment-priority classification
- Excess-reduction classification
- Recommended inventory actions

All checks must pass before the notebook is considered analytically complete.

In [24]:
# Inventory Optimization Reconciliation

reconciliation_checks = {

    "Optimization Base Record Count":
        len(optimization_base) == 3000,

    "Unique Product Count":
        optimization_base["product_id"].nunique() == 500,

    "Warehouse Count":
        optimization_base["warehouse_id"].nunique() == 6,

    "Inventory Value Reconciles":
        np.isclose(
            optimization_base["inventory_value"].sum(),
            285_985_182.04
        ),

    "Original Replenishment Gap Reconciles":
        np.isclose(
            optimization_base[
                "replenishment_gap_qty"
            ].sum(),
            14_788
        ),

    "Outstanding PO Quantity Reconciles":
        np.isclose(
            optimization_base[
                "outstanding_po_qty"
            ].sum(),
            132_826
        ),

    "Net Replenishment Quantity Reconciles":
        np.isclose(
            optimization_base[
                "net_replenishment_qty"
            ].sum(),
            1_984
        ),

    "Net Replenishment Value Reconciles":
        np.isclose(
            optimization_base[
                "net_replenishment_value"
            ].sum(),
            6_850_078.47,
            atol=0.01
        ),

    "Potential Excess Quantity Reconciles":
        np.isclose(
            optimization_base[
                "excess_inventory_qty"
            ].sum(),
            1_903.80,
            atol=0.01
        ),

    "Potential Excess Value Reconciles":
        np.isclose(
            optimization_base[
                "excess_inventory_value"
            ].sum(),
            6_631_536.78,
            atol=0.01
        ),

    "Optimization Status Complete":
        optimization_base[
            "optimization_status"
        ].notna().all(),

    "Optimization Status Count Reconciles":
        optimization_status_summary[
            "inventory_records"
        ].sum() == 3000,

    "ABC-XYZ Classification Complete":
        optimization_base[
            "abc_xyz_segment"
        ].notna().all(),

    "ABC-XYZ Product Count":
        abc_xyz_mapping[
            "product_id"
        ].nunique() == 500,

    "Replenishment Priority Count":
        replenishment_priority_summary[
            "inventory_records"
        ].sum() == 309,

    "Critical Replenishment Quantity":
        np.isclose(
            critical_replenishment[
                "net_replenishment_qty"
            ].sum(),
            834
        ),

    "Targeted Reduction Record Count":
        len(reduction_opportunities) == 104,

    "Recommended Action Complete":
        optimization_base[
            "recommended_action"
        ].notna().all(),

    "Recommended Action Count Reconciles":
        action_summary[
            "inventory_records"
        ].sum() == 3000
}


print("INVENTORY OPTIMIZATION RECONCILIATION\n")


for check, result in reconciliation_checks.items():

    print(
        f"{check:<45} : "
        f"{'PASS' if result else 'FAIL'}"
    )


inventory_optimization_reconciliation_passed = (
    all(reconciliation_checks.values())
)


print(
    "\nINVENTORY OPTIMIZATION RECONCILIATION STATUS :",
    "PASSED"
    if inventory_optimization_reconciliation_passed
    else "REVIEW REQUIRED"
)

INVENTORY OPTIMIZATION RECONCILIATION

Optimization Base Record Count                : PASS
Unique Product Count                          : PASS
Warehouse Count                               : PASS
Inventory Value Reconciles                    : PASS
Original Replenishment Gap Reconciles         : PASS
Outstanding PO Quantity Reconciles            : PASS
Net Replenishment Quantity Reconciles         : PASS
Net Replenishment Value Reconciles            : PASS
Potential Excess Quantity Reconciles          : PASS
Potential Excess Value Reconciles             : PASS
Optimization Status Complete                  : PASS
Optimization Status Count Reconciles          : PASS
ABC-XYZ Classification Complete               : PASS
ABC-XYZ Product Count                         : PASS
Replenishment Priority Count                  : PASS
Critical Replenishment Quantity               : PASS
Targeted Reduction Record Count               : PASS
Recommended Action Complete                   : PASS
Recomme

## Step 24 — Key Business Insights

### 1. Current Inventory Position

The inventory portfolio contains:

- **3,000 Product–Warehouse records**
- **500 unique products**
- **6 warehouses**
- Approximately **285.99 million** in total inventory value

Average inventory coverage is **32.18 days**, with a median of **31.17 days**.

---

### 2. Inventory Coverage Risk

Inventory coverage analysis identified:

- **240 records** with zero available inventory
- **174 records** with only 1–7 days of coverage
- **327 records** with 8–15 days of coverage
- **331 records** with more than 60 days of coverage

The 61+ day group holds approximately **59.07 million**, representing **20.65% of total inventory value**.

This indicates that inventory shortages and high inventory coverage exist simultaneously across different Product–Warehouse combinations.

---

### 3. Safety Stock Exposure

A total of **445 records (14.83%)** are currently below their defined safety-stock level.

These records represent approximately **5.73 million**, or only **2.00% of total inventory value**.

Although the financial exposure is relatively small, these positions may carry elevated service-level and stockout risk.

---

### 4. Reorder Requirement

Before considering inbound purchase orders:

- **1,311 records (43.70%)** have reached or fallen below their reorder point.
- **1,227 records** have a positive replenishment gap.
- The gross replenishment gap equals **14,788 units**.
- Estimated gross replenishment-gap value is approximately **48.26 million**.

However, this does not represent the true additional purchasing requirement because substantial inventory is already inbound.

---

### 5. Inbound Purchase Orders Significantly Reduce Shortage Exposure

Outstanding purchase orders contain **132,826 units of inbound inventory**.

After accounting for inbound supply:

- **918 shortage records are fully covered**
- Remaining replenishment-required records fall to **309**
- Net replenishment requirement falls to only **1,984 units**
- Remaining replenishment value is approximately **6.85 million**

This demonstrates why purchase-order visibility is essential when making replenishment decisions.

Ordering directly from the gross reorder signal could result in unnecessary additional inventory.

---

### 6. Current Optimization Position

The 3,000 Product–Warehouse records are classified as:

- **Balanced / Monitor:** 2,361 records (78.70%)
- **Potential Excess:** 330 records (11.00%)
- **Replenishment Required:** 309 records (10.30%)

Most inventory positions therefore do not currently require an immediate corrective action.

---

### 7. Potential Excess Inventory

Using 60 days of demand as an optimization screening benchmark:

- **330 records** contain potential excess inventory
- Potential excess quantity is approximately **1,903.80 units**
- Potential excess value is approximately **6.63 million**
- This represents **2.32% of total inventory value**

The 6.63 million should not automatically be interpreted as removable working capital because ABC–XYZ importance must also be considered.

---

### 8. Critical Replenishment Priority

ABC–XYZ prioritization identified **97 Critical Priority replenishment records**.

These require:

- **834 units**
- Approximately **3.79 million** in replenishment value
- **55.36% of the total remaining replenishment investment**

These shortages occur within financially important A-class products and should receive the strongest replenishment attention.

---

### 9. Replenishment Priority Distribution

The remaining 309 replenishment records are distributed as:

- **Critical Priority:** 97 records — 834 units — 3.79 million
- **High Priority:** 75 records — 501 units — 1.65 million
- **Moderate Priority:** 62 records — 413 units — 0.80 million
- **Low Priority:** 75 records — 236 units — 0.60 million

This demonstrates that replenishment decisions should be prioritized rather than treating every shortage equally.

---

### 10. Excess Inventory Requires Selective Reduction

ABC–XYZ analysis shows that much of the potential excess inventory belongs to strategically important products.

Approximately **79.67% of potential excess value** falls into either:

- Low Reduction Priority
- Protect / Review Carefully

Therefore, reducing the entire 6.63 million of identified excess inventory could create unnecessary service-level risk.

---

### 11. Strongest Inventory-Reduction Opportunities

The strongest reduction candidates are products classified under:

- Highest Reduction Priority
- High Reduction Priority

Together these represent:

- **104 Product–Warehouse records**
- Approximately **299.60 excess units**
- Approximately **610,994 in excess inventory value**
- **9.21% of total potential excess value**

These records provide a more defensible starting point for working-capital reduction than applying broad inventory cuts.

---

### 12. Category-Level Optimization

**Electronics** has the largest remaining replenishment exposure at approximately **2.15 million**.

Because Electronics also represents the largest inventory category by value in the broader inventory analysis, replenishment decisions within this category deserve particular attention.

Potential excess inventory is present across all five product categories, indicating that inventory optimization is not isolated to a single business area.

---

### 13. Recommended Inventory Actions

The optimization model translates analytical findings into operational actions including:

- **Urgent Replenishment**
- **High-Priority Replenishment**
- **Planned Replenishment**
- **Low-Priority Replenishment**
- **Reduce / Rationalize Stock**
- **Reduce Excess Stock**
- **Review Excess Stock**
- **Monitor Excess Stock**
- **Protect / Review Excess**
- **Maintain / Monitor**

All **3,000 Product–Warehouse records** received a recommended inventory action.

---

## Management Takeaway

The inventory portfolio does not primarily suffer from a simple company-wide shortage or overstock problem. Instead, it contains **localized inventory imbalances across products and warehouses**.

The gross reorder signal suggests a requirement of **14,788 units**, but after considering existing inbound purchase orders, the true additional requirement falls dramatically to **1,984 units**.

This highlights the risk of replenishing inventory without considering the supply pipeline.

At the same time, approximately **6.63 million** of inventory exceeds the 60-day coverage benchmark. However, ABC–XYZ prioritization shows that only a smaller portion should receive aggressive reduction attention.

The strongest inventory strategy is therefore to:

1. Prioritize the **97 critical replenishment records**
2. Avoid duplicate ordering where inbound POs already cover shortages
3. Target the strongest **104 excess-reduction opportunities**
4. Protect strategically important high-value inventory
5. Maintain the **2,361 currently balanced Product–Warehouse positions**

This targeted approach can improve product availability while controlling inventory investment and reducing unnecessary working-capital exposure.

## Step 25 — Final Inventory Optimization Validation

Perform the final validation of the Inventory Optimization notebook.

The validation confirms that:

- All 3,000 Product–Warehouse inventory records are preserved
- All 500 products and 6 warehouses are represented
- Inventory value reconciles with the source inventory dataset
- Coverage calculations are complete
- Safety-stock and reorder calculations are complete
- Outstanding purchase-order quantities are incorporated
- Net replenishment requirements reconcile
- Potential excess inventory calculations reconcile
- ABC–XYZ classifications are complete
- Optimization statuses and priorities are complete
- Every Product–Warehouse record has a recommended inventory action
- Step 23 reconciliation passed

This ensures that the optimization model is internally consistent and ready for downstream reporting and decision support.

In [25]:
# Final Inventory Optimization Validation

valid_segments = {
    "AX", "AY", "AZ",
    "BX", "BY", "BZ",
    "CX", "CY", "CZ"
}


valid_optimization_statuses = {
    "Balanced / Monitor",
    "Potential Excess",
    "Replenishment Required"
}


final_validation_checks = {

    "Inventory Record Count":
        len(optimization_base) == 3000,

    "Unique Product Count":
        optimization_base["product_id"].nunique() == 500,

    "Warehouse Count":
        optimization_base["warehouse_id"].nunique() == 6,

    "Inventory Value Reconciles":
        np.isclose(
            optimization_base["inventory_value"].sum(),
            285_985_182.04
        ),

    "Coverage Calculation Complete":
        optimization_base[
            "coverage_days"
        ].notna().all(),

    "Safety Stock Status Complete":
        optimization_base[
            "safety_stock_status"
        ].notna().all(),

    "Reorder Status Complete":
        optimization_base[
            "reorder_status"
        ].notna().all(),

    "Gross Replenishment Gap Reconciles":
        np.isclose(
            optimization_base[
                "replenishment_gap_qty"
            ].sum(),
            14_788
        ),

    "Outstanding PO Quantity Reconciles":
        np.isclose(
            optimization_base[
                "outstanding_po_qty"
            ].sum(),
            132_826
        ),

    "Net Replenishment Quantity Reconciles":
        np.isclose(
            optimization_base[
                "net_replenishment_qty"
            ].sum(),
            1_984
        ),

    "Net Replenishment Value Reconciles":
        np.isclose(
            optimization_base[
                "net_replenishment_value"
            ].sum(),
            6_850_078.47,
            atol=0.01
        ),

    "Potential Excess Quantity Reconciles":
        np.isclose(
            optimization_base[
                "excess_inventory_qty"
            ].sum(),
            1_903.80,
            atol=0.01
        ),

    "Potential Excess Value Reconciles":
        np.isclose(
            optimization_base[
                "excess_inventory_value"
            ].sum(),
            6_631_536.78,
            atol=0.01
        ),

    "Optimization Status Complete":
        optimization_base[
            "optimization_status"
        ].notna().all(),

    "Optimization Status Domain Valid":
        set(
            optimization_base[
                "optimization_status"
            ].unique()
        ).issubset(
            valid_optimization_statuses
        ),

    "ABC-XYZ Classification Complete":
        optimization_base[
            "abc_xyz_segment"
        ].notna().all(),

    "ABC-XYZ Segment Domain Valid":
        set(
            optimization_base[
                "abc_xyz_segment"
            ].unique()
        ).issubset(
            valid_segments
        ),

    "ABC-XYZ Product Count":
        abc_xyz_mapping[
            "product_id"
        ].nunique() == 500,

    "Replenishment Required Count":
        (
            optimization_base[
                "net_replenishment_qty"
            ] > 0
        ).sum() == 309,

    "Potential Excess Count":
        (
            optimization_base[
                "excess_inventory_qty"
            ] > 0
        ).sum() == 330,

    "Critical Priority Count":
        (
            optimization_base[
                "replenishment_priority"
            ] == "Critical Priority"
        ).sum() == 97,

    "Targeted Reduction Count":
        len(
            reduction_opportunities
        ) == 104,

    "Recommended Actions Complete":
        optimization_base[
            "recommended_action"
        ].notna().all(),

    "Recommended Action Count":
        action_summary[
            "inventory_records"
        ].sum() == 3000,

    "Step 23 Reconciliation Passed":
        inventory_optimization_reconciliation_passed
}


print("FINAL INVENTORY OPTIMIZATION VALIDATION\n")


for check, result in final_validation_checks.items():

    print(
        f"{check:<45} : "
        f"{'PASS' if result else 'FAIL'}"
    )


final_inventory_optimization_status = (
    all(final_validation_checks.values())
)


print(
    "\nFINAL INVENTORY OPTIMIZATION STATUS :",
    "PASSED"
    if final_inventory_optimization_status
    else "REVIEW REQUIRED"
)

FINAL INVENTORY OPTIMIZATION VALIDATION

Inventory Record Count                        : PASS
Unique Product Count                          : PASS
Warehouse Count                               : PASS
Inventory Value Reconciles                    : PASS
Coverage Calculation Complete                 : PASS
Safety Stock Status Complete                  : PASS
Reorder Status Complete                       : PASS
Gross Replenishment Gap Reconciles            : PASS
Outstanding PO Quantity Reconciles            : PASS
Net Replenishment Quantity Reconciles         : PASS
Net Replenishment Value Reconciles            : PASS
Potential Excess Quantity Reconciles          : PASS
Potential Excess Value Reconciles             : PASS
Optimization Status Complete                  : PASS
Optimization Status Domain Valid              : PASS
ABC-XYZ Classification Complete               : PASS
ABC-XYZ Segment Domain Valid                  : PASS
ABC-XYZ Product Count                         : PASS
Reple